# 📘 智能体架构 14：可观测性 + 演练（Dry-Run）支架

本系列中至关重要的一节，聚焦 AI 智能体的部署与运行安全。我们将实现 **可观测性与演练（Dry-Run）支架**，这是测试、调试与安全管理与现实世界交互的智能体的必备模式。

核心原则简单而有力：**在确切知道智能体将要做什么之前，切勿在真实环境中执行其动作。** 该架构将「三思而后行」正式化：智能体首先在 `dry_run`（演练）模式下执行计划，不改变真实世界，但会生成详细日志与清晰行动计划。该计划随后呈现给人类（或自动检查器）审批，之后才允许最终的真实执行。

为此，我们将构建 **企业社交媒体智能体**，负责创建与发布帖子。你会看到演练支架如何支持：
1.  **生成拟发布内容：** 根据提示词创意起草帖子。
2.  **执行演练：** 智能体在沙箱中以 `dry_run=True` 调用 `publish`，记录「将会」发生什么。
3.  **人在回路审阅：** 操作员看到完整帖子与演练轨迹，须输入 `approve` 才能继续。
4.  **真实执行：** 仅在批准后再以 `dry_run=False` 调用 `publish`，执行真实动作。

该模式是负责任部署智能体的基石，为在生产环境中安全运行 AI 提供所需的透明度与控制力。


### 定义
**可观测性与演练（Dry-Run）支架** 是一种测试与部署架构：拦截智能体的动作，先在「演练」或「沙箱」模式下执行，模拟动作而不产生真实世界影响；将得到的计划与日志呈现供审阅；仅在明确批准后，才在真实环境中执行。

### 高层工作流

1.  **智能体提出动作：** 智能体确定要执行的计划或具体工具调用（例如 `api.post_update(...)`）。
2.  **演练执行：** 支架以 `dry_run=True` 调用智能体的计划；底层工具识别该标志，只输出它们 *将会* 做什么，并附带日志与追踪。
3.  **收集可观测数据：** 支架捕获拟议动作、演练日志以及仿真中的其他相关追踪数据。
4.  **人工/自动审阅：** 将可观测数据交给审阅者。人类可检查正确性、安全性与目标一致性；自动系统可检查策略违规或已知不良模式。
5.  **放行/否决：** 审阅者做出 `approve` 或 `reject` 决策。
6.  **真实执行（放行时）：** 若批准，支架再次执行智能体的动作，此时 `dry_run=False`，产生真实世界影响。

### 适用场景
*   **调试与测试：** 在开发中精确了解智能体如何理解任务、采取哪些动作，且无副作用。
*   **生产校验与安全：** 对任何能改状态、花钱、发通信或执行其他不可逆动作的智能体，作为生产中的常驻能力。
*   **智能体的 CI/CD：** 将演练支架纳入自动化测试流水线，在部署新版本前校验行为。

### 优势与局限
*   **优势：**
    *   **透明度与安全上限：** 提供可审计的「预览」，避免代价高昂或令人尴尬的错误。
    *   **极利于调试：** 易于追踪逻辑与工具调用，无需撤销真实世界变更。
*   **局限：**
    *   **拖慢部署/执行：** 强制审阅（尤其人工）带来延迟，不适合强实时场景。
    *   **依赖工具支持：** 智能体使用的工具与 API 必须支持 `dry_run` 模式。


## 阶段 0：基础与配置

库与环境变量的常规配置。


In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv

In [2]:
import os
import datetime
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Dry-Run Harness (DeepSeek)"

required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：环境与工具

该架构的核心是支持 `dry_run` 模式的工具。我们将实现一个简单的 `SocialMediaAPI` 类；其 `publish_post` 方法根据 `dry_run` 标志表现不同，从而提供所需的可观测性。


In [3]:
console = Console()

# Structured model for the agent's proposed post
class SocialMediaPost(BaseModel):
    content: str = Field(description="The full text content of the social media post.")
    hashtags: List[str] = Field(description="A list of relevant hashtags, without the '#'.")

# The key component: A tool with a dry_run flag
class SocialMediaAPI:
    """A mock social media API that supports a dry-run mode."""
    
    def publish_post(self, post: SocialMediaPost, dry_run: bool = True) -> Dict[str, Any]:
        """Publishes a post to the social media feed."""
        timestamp = datetime.datetime.now().isoformat()
        hashtags_str = ' '.join([f'#{h}' for h in post.hashtags])
        full_post_text = f"{post.content}\n\n{hashtags_str}"
        
        if dry_run:
            # In dry-run mode, we don't execute, we just return the plan and logs
            log_message = f"[DRY RUN] At {timestamp}, would publish the following post:\n--- PREVIEW ---\n{full_post_text}\n--- END PREVIEW ---"
            console.print(Panel(log_message, title="[yellow]Dry Run Log[/yellow]", border_style="yellow"))
            return {"status": "DRY_RUN_SUCCESS", "log": log_message, "proposed_post": full_post_text}
        else:
            # In live mode, we execute the action
            log_message = f"[LIVE] At {timestamp}, successfully published post!"
            console.print(Panel(log_message, title="[green]Live Execution Log[/green]", border_style="green"))
            # Here you would have the actual API call, e.g., twitter_client.create_tweet(...)
            return {"status": "LIVE_SUCCESS", "log": log_message, "post_id": f"post_{hash(full_post_text)}"}

social_media_tool = SocialMediaAPI()
print("Dry-run capable SocialMediaAPI tool defined successfully.")

Dry-run capable SocialMediaAPI tool defined successfully.


## 阶段 2：用 LangGraph 搭建演练支架

接下来构建完整工作流。图将管理过程状态：从提出动作，到演练与审阅，再到基于人工批准的条件执行。


In [4]:
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0.5,
)

# LangGraph State
class AgentState(TypedDict):
    user_request: str
    proposed_post: Optional[SocialMediaPost]
    dry_run_log: Optional[str]
    review_decision: Optional[str] # 'approve' or 'reject'
    final_status: str

# Graph Nodes
def propose_post_node(state: AgentState) -> Dict[str, Any]:
    """The creative agent that drafts the social media post."""
    console.print("--- 📝 Social Media Agent Drafting Post ---")
    prompt = ChatPromptTemplate.from_template(
        "You are a creative and engaging social media manager for a major AI company. Based on the user's request, draft a compelling social media post, including relevant hashtags.\n\nRequest: {request}"
    )
    post_generator_llm = llm.with_structured_output(SocialMediaPost)
    chain = prompt | post_generator_llm
    proposed_post = chain.invoke({"request": state['user_request']})
    return {"proposed_post": proposed_post}

def dry_run_review_node(state: AgentState) -> Dict[str, Any]:
    """Performs the dry run and prompts for human review."""
    console.print("--- 🧐 Performing Dry Run & Awaiting Human Review ---")
    dry_run_result = social_media_tool.publish_post(state['proposed_post'], dry_run=True)
    
    # Present the plan for review
    review_panel = Panel(
        f"[bold]Proposed Post:[/bold]\n{dry_run_result['proposed_post']}",
        title="[bold yellow]Human-in-the-Loop: Review Required[/bold yellow]",
        border_style="yellow"
    )
    console.print(review_panel)
    
    # Get human approval
    decision = ""
    while decision.lower() not in ["approve", "reject"]:
        decision = console.input("Type 'approve' to publish or 'reject' to cancel: ")
        
    return {"dry_run_log": dry_run_result['log'], "review_decision": decision.lower()}

def execute_live_post_node(state: AgentState) -> Dict[str, Any]:
    """Executes the live post after approval."""
    console.print("--- ✅ Post Approved, Executing Live ---")
    live_result = social_media_tool.publish_post(state['proposed_post'], dry_run=False)
    return {"final_status": f"Post successfully published! ID: {live_result.get('post_id')}"}

def post_rejected_node(state: AgentState) -> Dict[str, Any]:
    """Handles the case where the post is rejected."""
    console.print("--- ❌ Post Rejected by Human Reviewer ---")
    return {"final_status": "Action was rejected by the reviewer and not executed."}

# Conditional Edge
def route_after_review(state: AgentState) -> str:
    """Routes to execution or rejection based on the human review."""
    return "execute_live" if state["review_decision"] == "approve" else "reject"

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("propose_post", propose_post_node)
workflow.add_node("dry_run_review", dry_run_review_node)
workflow.add_node("execute_live", execute_live_post_node)
workflow.add_node("reject", post_rejected_node)

workflow.set_entry_point("propose_post")
workflow.add_edge("propose_post", "dry_run_review")
workflow.add_conditional_edges("dry_run_review", route_after_review, {"execute_live": "execute_live", "reject": "reject"})
workflow.add_edge("execute_live", END)
workflow.add_edge("reject", END)

dry_run_agent = workflow.compile()
print("Dry-Run Harness agent graph compiled successfully.")

Dry-Run Harness agent graph compiled successfully.


## 阶段 3：演示

测试完整系统：先使用我们会批准的、安全的常规请求；再使用可能生成风险帖子的模糊请求，我们将予以拒绝。


In [5]:
def run_agent_with_harness(request: str):
    initial_state = {"user_request": request}
    # Note: You will be prompted to type in the console below the cell.
    result = dry_run_agent.invoke(initial_state)
    console.print(f"\n[bold]Final Status:[/bold] {result['final_status']}")

# Test 1: A safe post that we will approve.
console.print("--- ✅ Test 1: Safe Post (Approve) ---")
run_agent_with_harness("Draft a positive launch announcement for our new AI model, 'Nebula'.")

# Test 2: A risky post that we will reject.
console.print("\n--- ❌ Test 2: Risky Post (Reject) ---")
run_agent_with_harness("Draft a post that emphasizes how much better our new 'Nebula' model is than the competition.")

--- ✅ Test 1: Safe Post (Approve) ---


--- 📝 Social Media Agent Drafting Post ---
--- 🧐 Performing Dry Run & Awaiting Human Review ---


                             Dry Run Log                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ [DRY RUN] At 2024-06-25T12:00:00.000000, would publish the        ┃
┃ following post:                                                  ┃
┃ --- PREVIEW ---                                                  ┃
┃ We're thrilled to announce the launch of our new flagship AI     ┃
┃ model, 'Nebula'! It's set to revolutionize natural language      ┃
┃ understanding and generation. A new era of AI is here!           ┃
┃                                                                  ┃
┃ #AI #Innovation #LaunchDay #Tech #Nebula                         ┃
┃ --- END PREVIEW ---                                              ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Human-in-the-Loop: Review Required                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Proposed Post:                                                   ┃
┃ We're thrilled to announce the launch of our new flagship AI     ┃
┃ model, 'Nebula'! It's set to revolutionize natural language      ┃
┃ understanding and generation. A new era of AI is here!           ┃
┃                                                                  ┃
┃ #AI #Innovation #LaunchDay #Tech #Nebula                         ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛Type 'approve' to publish or 'reject' to cancel: 

--- ✅ Post Approved, Executing Live ---


...

--- 📝 Social Media Agent Drafting Post ---
--- 🧐 Performing Dry Run & Awaiting Human Review ---


                             Dry Run Log                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ [DRY RUN] At 2024-06-25T12:00:01.000000, would publish the        ┃
┃ following post:                                                  ┃
┃ --- PREVIEW ---                                                  ┃
┃ Our new 'Nebula' AI is so advanced, it's basically going to      ┃
┃ make all our competitors obsolete. They just can't keep up.      ┃
┃ Get ready for the future.                                        ┃
┃                                                                  ┃
┃ #GameChanger #AI #Disruption #NoCompetition #FutureIsNow         ┃
┃ --- END PREVIEW ---                                              ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

                   Human-in-the-Loop: Review Required                   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Proposed Post:                                                   ┃
┃ Our new 'Nebula' AI is so advanced, it's basically going to      ┃
┃ make all our competitors obsolete. They just can't keep up.      ┃
┃ Get ready for the future.                                        ┃
┃                                                                  ┃
┃ #GameChanger #AI #Disruption #NoCompetition #FutureIsNow         ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛Type 'approve' to publish or 'reject' to cancel: 

--- ❌ Post Rejected by Human Reviewer ---



Final Status: Action was rejected by the reviewer and not executed.


### 结果分析

演示充分展示了支架的价值：

1.  **安全帖子：** 第一个请求很直接。智能体生成了专业且热情的帖子。演练准确预览了将要发布的内容。我们批准后，`[LIVE]` 日志确认已执行真实动作。流程按设计工作。

2.  **风险帖子：** 第二个请求较模糊，可能被解读为攻击性。智能体在强调「更胜一筹」的提示下，起草了傲慢且不专业的帖子（例如让竞争对手「全部过时」）。虽然满足了创意提示，但并非真实公司愿意发布的信息。

此处支架发挥作用：演练在内容 *可能* 被发布之前就暴露了风险。人工审阅者轻易识别不当语气并输入 `reject`。图正确路由到 `post_rejected_node`，最终状态确认 **未执行真实动作。** 一次简单的结构化工作流就避免了一场潜在公关危机。

这清晰分离了智能体创意但不可预测的生成，与确定性、受控的执行，提供了关键安全层。


## 小结

本笔记本中我们构建了完整的 **可观测性与演练（Dry-Run）支架**。该架构不仅是功能，更是智能体与现实世界交互的部署哲学。通过强制「提出 → 审阅 → 执行」循环，我们获得：

- **透明度：** 在动作发生前确切知道智能体意图。
- **控制：** 人在回路（或自动规则引擎）对任何动作拥有最终否决权。
- **安全：** 避免意外、昂贵或有害的行为，从「希望式执行」走向「有把握部署」。

该模式会引入延迟，但对多数真实应用而言，其带来的安全与可靠性不可或缺。它是构建稳健、可信、可投产智能体系统的必备工具。
